# Merge count files

In [2]:
import pandas as pd
import glob
import os

# Folder containing all count files
input_dir = "./"

# Get all counts files
files = glob.glob(os.path.join(input_dir, "*_counts.txt"))

merged = None

for file in files:

    # Read file, skipping the first line (Program:...)
    df = pd.read_csv(file, sep="\t", skiprows=1)

    # Keep Geneid + last column (count column)
    df = df.iloc[:, [0, -1]]

    # Rename last column using sample name
    sample_name = os.path.basename(file).replace("_counts.txt", "")
    df.columns = ["Geneid", sample_name]

    # Merge based on Geneid
    if merged is None:
        merged = df
    else:
        merged = pd.merge(
            merged,
            df,
            on="Geneid",
            how="outer"
        )

# Save merged file
output_file = os.path.join(input_dir, "merged_counts.csv")
merged.to_csv(output_file, index=False)

print("Done!")
print("Output:", output_file)

Done!
Output: ./merged_counts.csv


#  Each strain or dataset wise if gene comes zero all the samples belong to same strain means remove that gene row

In [15]:
import pandas as pd

counts_file = "merged_counts.csv"
metadata_file = "metadata.final.csv"

output_file = "merged_counts_strain_filtered.csv"

# Read files
counts = pd.read_csv(counts_file)
metadata = pd.read_csv(metadata_file, encoding="latin1")

# Sample columns from counts file
sample_columns = counts.columns[1:]

# Make sure sample IDs are strings
metadata["Run"] = metadata["Run"].astype(str)

# Sample -> Strain mapping
sample_to_strain = dict(
    zip(metadata["Run"], metadata["Strain"])
)

# Group samples by strain
strain_samples = {}

for sample in sample_columns:

    if sample in sample_to_strain:

        strain = sample_to_strain[sample]

        strain_samples.setdefault(strain, []).append(sample)


# Print strain groups
print("\n========== STRAIN GROUPS ==========")

for strain, samples in strain_samples.items():
    print(f"{strain}: {samples}")


# ------------------------------------------------
# FILTER GENES
# ------------------------------------------------

keep_genes = []

for _, row in counts.iterrows():

    remove_gene = False

    # Check each strain
    for strain, samples in strain_samples.items():

        # Counts for this gene in all samples of this strain
        values = pd.to_numeric(row[samples])

        # If ALL samples of this strain are zero
        if (values == 0).all():

            remove_gene = True

            print(
                f"Removing {row['Gene']} "
                f"because all samples in {strain} are zero"
            )

            break

    keep_genes.append(not remove_gene)


# Apply filtering
filtered_counts = counts[keep_genes].copy()


# Save output
filtered_counts.to_csv(
    output_file,
    index=False
)


# Summary
print("\n========== SUMMARY ==========")

print("Original genes :", len(counts))
print("Remaining genes:", len(filtered_counts))
print("Removed genes  :", len(counts) - len(filtered_counts))

print("\nOutput:", output_file)


========== STRAIN GROUPS ==========
Acinetobacter baumannii 34654: ['SRR1184547', 'SRR1184575', 'SRR1184593', 'SRR1184594', 'SRR1184598', 'SRR1184599', 'SRR1184630', 'SRR1184649', 'SRR1184672', 'SRR1184651']
Acinetobacter baumannii 1428368: ['SRR1185425', 'SRR1185426', 'SRR1185427', 'SRR1185431', 'SRR1185433', 'SRR1185436', 'SRR1185437', 'SRR1185440', 'SRR1185441', 'SRR1185447']
Acinetobacter baumannii 1207552: ['SRR1185450', 'SRR1185451', 'SRR1185452', 'SRR1185455', 'SRR1185456', 'SRR1185503', 'SRR1185504', 'SRR1185507', 'SRR1185508', 'SRR1185512']
Acinetobacter baumannii 1457504: ['SRR1185891', 'SRR1185892', 'SRR1185893', 'SRR1185896', 'SRR1185897', 'SRR1185900', 'SRR1185901', 'SRR1185904', 'SRR1185908', 'SRR1185913']
Acinetobacter baumannii ATCC 19606 = CIP 70.34 = JCM 6841: ['SRR1634081', 'SRR1634082', 'SRR1634083', 'SRR1634084', 'SRR1634089', 'SRR1634090', 'SRR1634091', 'SRR1634092', 'SRR1634093', 'SRR1634094', 'SRR1634095', 'SRR1634096', 'SRR1634097', 'SRR1634098', 'SRR1634099',

# Grouping the sample based on the GEO ID

In [18]:
import pandas as pd
import os
import re

# --------------------------------------------------
# INPUT FILES
# --------------------------------------------------

COUNTS_FILE = "merged_counts_strain_filtered.csv"
META_FILE = "metadata.final.csv"

OUTPUT_DIR = "GEO_grouped"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# --------------------------------------------------
# READ FILES
# --------------------------------------------------

counts = pd.read_csv(COUNTS_FILE)
metadata = pd.read_csv(META_FILE, encoding="latin1")



# Count matrix sample names
sample_columns = [
    c for c in counts.columns
    if c != "Gene"
]

sample_columns_clean = {
    c: clean_run(c)
    for c in sample_columns
}


# --------------------------------------------------
# CREATE RUN -> GEO ID MAPPING
# --------------------------------------------------

run_to_geo = dict(
    zip(
        metadata["Run"],
        metadata["GEO ID"]
    )
)


# --------------------------------------------------
# GROUP SAMPLES BY GEO ID
# --------------------------------------------------

geo_samples = {}

for column, run in sample_columns_clean.items():

    if run in run_to_geo:

        geo = run_to_geo[run]

        geo_samples.setdefault(geo, []).append(column)

    else:

        print(
            f"[WARNING] {run} not found in metadata"
        )


# --------------------------------------------------
# SAVE ONE FILE PER GEO ID
# --------------------------------------------------

for geo, sample_cols in geo_samples.items():

    # Gene + samples from this GEO
    sub = counts[
        ["Gene"] + sample_cols
    ]

    output_file = os.path.join(
        OUTPUT_DIR,
        f"{geo}_counts.csv"
    )

    sub.to_csv(
        output_file,
        index=False
    )

    print(
        f"[SAVED] {output_file} "
        f"→ {len(sample_cols)} samples"
    )


# --------------------------------------------------
# SUMMARY
# --------------------------------------------------

print("\n========== SUMMARY ==========")

print("Total GEO datasets:", len(geo_samples))

for geo, samples in geo_samples.items():

    print(
        f"{geo}: {len(samples)} samples"
    )

[SAVED] GEO_grouped\GSE56218_counts.csv → 10 samples
[SAVED] GEO_grouped\GSE56223_counts.csv → 10 samples
[SAVED] GEO_grouped\GSE56222_counts.csv → 10 samples
[SAVED] GEO_grouped\GSE56224_counts.csv → 10 samples
[SAVED] GEO_grouped\GSE62794_counts.csv → 20 samples
[SAVED] GEO_grouped\GSE131451_counts.csv → 6 samples
[SAVED] GEO_grouped\GSE190441_counts.csv → 18 samples
[SAVED] GEO_grouped\GSE218219_counts.csv → 12 samples

========== SUMMARY ==========
Total GEO datasets: 8
GSE56218: 10 samples
GSE56223: 10 samples
GSE56222: 10 samples
GSE56224: 10 samples
GSE62794: 20 samples
GSE131451: 6 samples
GSE190441: 18 samples
GSE218219: 12 samples
